# Phase 5 — Read-Only Dashboard

This dashboard subscribes to MQTT topics and visualizes live simulation data.

Read-only behavior:
- Subscribes to state, match, and switch topics
- Computes live metrics
- Renders map markers when `anymap-ts` is available
- Does not publish control commands

In [1]:
from collections import Counter
from datetime import datetime, timezone
import json
from pathlib import Path
import sys
import time

cwd = Path.cwd()
candidate_paths = [cwd / "src", cwd.parent / "src"]
for path in candidate_paths:
    if path.exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))

from IPython.display import display
from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector
import yaml

In [2]:
app_cfg = load_config()

config_path = Path("config.yaml")
if not config_path.exists():
    config_path = Path("../config.yaml")

raw_cfg = {}
if config_path.exists():
    raw_cfg = yaml.safe_load(config_path.read_text(encoding="utf-8")) or {}

sim_cfg = raw_cfg.get("dating_simulation", {})
BAR_IDS = sim_cfg.get("bar_ids", ["bar_1", "bar_2", "bar_3", "bar_4"])

TOPIC_BASE = app_cfg.mqtt.base_topic
TOPIC_STATE = f"{TOPIC_BASE}/agents/state"
TOPIC_MATCH = f"{TOPIC_BASE}/events/match"
TOPIC_SWITCH = f"{TOPIC_BASE}/events/switch"

print({
    "topic_state": TOPIC_STATE,
    "topic_match": TOPIC_MATCH,
    "topic_switch": TOPIC_SWITCH,
    "bars": BAR_IDS,
})

{'topic_state': 'simulated-city/agents/state', 'topic_match': 'simulated-city/events/match', 'topic_switch': 'simulated-city/events/switch', 'bars': ['bar_1', 'bar_2', 'bar_3', 'bar_4']}


In [3]:
map_widget = None
marker_ids = set()

try:
    from simulated_city.maplibre_live import LiveMapLibreMap
    map_widget = LiveMapLibreMap(center=(12.5588, 55.6699), zoom=15.9, height="550px", width="100%")
    display(map_widget)
except Exception as exc:
    print(f"Map disabled ({exc})")

In [ ]:
latest_people = {}
match_events = []
switch_events = []

BAR_SIZE_M = float(sim_cfg.get("bar_size_m", 100.0))

def parse_timestamp(value):
    if not value or not isinstance(value, str):
        return None
    try:
        return datetime.fromisoformat(value.replace("Z", "+00:00"))
    except ValueError:
        return None

def latest_people_state():
    return dict(latest_people)

def xy_to_kodbyen_lnglat(x, y, width=100.0):
    min_lng, max_lng = 12.5480, 12.5645
    min_lat, max_lat = 55.6650, 55.6745
    xn = max(0.0, min(1.0, x / width))
    yn = max(0.0, min(1.0, y / width))
    lng = min_lng + xn * (max_lng - min_lng)
    lat = min_lat + yn * (max_lat - min_lat)
    return lng, lat

def marker_color_for_state(state: str) -> str:
    if state == "matched":
        return "green"
    if state == "talking":
        return "yellow"
    return "blue"

def refresh_map(snapshot=None):
    global map_widget, marker_ids
    if map_widget is None:
        return

    if snapshot is None:
        snapshot = latest_people_state()

    active_marker_ids = set()
    for pid, data in snapshot.items():
        state = data.get("state", "idle")
        if state == "removed":
            continue

        lng, lat = xy_to_kodbyen_lnglat(
            data.get("x", 0.0),
            data.get("y", 0.0),
            BAR_SIZE_M,
        )
        color = marker_color_for_state(state)
        popup = f"{pid} | {state} | {data.get('color', '?')}"
        marker_id = f"p-{pid}"
        active_marker_ids.add(marker_id)

        map_widget.move_marker(
            marker_id,
            (lng, lat),
            color=color,
            popup=popup,
        )

    stale_marker_ids = marker_ids - active_marker_ids
    for marker_id in stale_marker_ids:
        map_widget.remove_marker(marker_id)

    marker_ids = active_marker_ids

def summary(snapshot):
    states = [p.get("state", "idle") for p in snapshot.values()]
    present = sum(1 for state in states if state != "removed")
    matched = sum(1 for state in states if state == "matched")
    return {
        "present": present,
        "matched": matched,
        "switches": len(switch_events),
        "events_seen": len(snapshot),
    }

def print_dashboard_summary():
    snapshot = latest_people_state()
    kpi = summary(snapshot)
    print(
        f"present={kpi['present']} matched={kpi['matched']} switches={kpi['switches']} events={kpi['events_seen']}",
        flush=True,
    )

In [5]:
connector = MqttConnector(app_cfg.mqtt, client_id_suffix="dashboard")

def on_message(client, userdata, message):
    try:
        payload = json.loads(message.payload.decode("utf-8"))
    except Exception:
        return

    if message.topic == TOPIC_STATE:
        person_id = payload.get("person_id")
        if person_id is not None:
            latest_people[person_id] = payload
    elif message.topic == TOPIC_MATCH:
        ts = parse_timestamp(payload.get("timestamp"))
        if ts is not None and (datetime.now(timezone.utc) - ts).total_seconds() > 5:
            return
        match_events.append(payload)
    elif message.topic == TOPIC_SWITCH:
        switch_events.append(payload)

connector.client.on_message = on_message

connector.connect()
if connector.wait_for_connection(timeout=3.0):
    connector.client.subscribe(TOPIC_STATE)
    connector.client.subscribe(TOPIC_MATCH)
    connector.client.subscribe(TOPIC_SWITCH)
    print("Dashboard subscribed to state/match/switch topics.")
else:
    print("Dashboard did not connect in time.")

Dashboard subscribed to state/match/switch topics.


In [6]:
DASHBOARD_RUN_SECONDS = 300
REFRESH_EVERY_SECONDS = 5

run_started = time.monotonic()
while (time.monotonic() - run_started) < DASHBOARD_RUN_SECONDS:
    time.sleep(REFRESH_EVERY_SECONDS)
    print_dashboard_summary()
    refresh_map()

print("Dashboard 5-minute run complete.")

present=50 matched=0 switches=0 events=50
present=50 matched=0 switches=0 events=50
present=50 matched=0 switches=0 events=50
present=50 matched=0 switches=0 events=50
present=50 matched=0 switches=0 events=50
present=50 matched=0 switches=0 events=50
present=50 matched=0 switches=0 events=50
present=50 matched=0 switches=0 events=50
present=50 matched=0 switches=0 events=50
present=50 matched=0 switches=0 events=50
present=50 matched=4 switches=0 events=50
present=50 matched=4 switches=0 events=50
present=50 matched=4 switches=0 events=50
present=46 matched=0 switches=0 events=50
present=46 matched=0 switches=0 events=50
present=46 matched=0 switches=0 events=50
present=46 matched=0 switches=0 events=50
present=46 matched=0 switches=0 events=50
present=46 matched=0 switches=0 events=50
present=46 matched=0 switches=0 events=50
present=46 matched=0 switches=0 events=50
present=46 matched=2 switches=0 events=50
present=46 matched=2 switches=0 events=50
present=46 matched=2 switches=0 ev

present=50 matched=0 switches=0 events=50
present=50 matched=0 switches=0 events=50
present=50 matched=0 switches=0 events=50
present=50 matched=0 switches=0 events=50
present=50 matched=0 switches=0 events=50
present=50 matched=0 switches=0 events=50
present=50 matched=0 switches=0 events=50
present=50 matched=0 switches=0 events=50
present=50 matched=0 switches=0 events=50
present=50 matched=0 switches=0 events=50
present=50 matched=4 switches=0 events=50
present=50 matched=4 switches=0 events=50
present=50 matched=4 switches=0 events=50
present=46 matched=0 switches=0 events=50
present=46 matched=0 switches=0 events=50
present=46 matched=0 switches=0 events=50
present=46 matched=0 switches=0 events=50
present=46 matched=0 switches=0 events=50
present=46 matched=0 switches=0 events=50
present=46 matched=0 switches=0 events=50
present=46 matched=0 switches=0 events=50
present=46 matched=2 switches=0 events=50
present=46 matched=2 switches=0 events=50
present=46 matched=2 switches=0 ev

KeyboardInterrupt: 

In [ ]:
connector.disconnect()
print("Dashboard disconnected.")